<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d4-segmentation-starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 · Notebook 8 — Project II starter: finding and counting cells

In [ ]:
# Installs what Colab doesn't ship with.
!pip install -q cellpose<4

On On Day 2 you counted nuclei with a threshold and a blob count, and every
pair got a different number. Project II asks for the same thing done well:
find every cell in huge stained biopsy images, count them, and draw an
outline (a **mask**) around each one. This notebook does a first version with
a model that was trained by other scientists to find nuclei, and shows you
the two problems Project II teams always meet: how to score a mask, and what
happens at the edges of tiles.

Same rules: 🔮 predict, ✏️ replace the `...`, ✅ check. Swap driver and
navigator at every `##`.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from scipy import ndimage


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_model(name, *paths):
    """Stop with a plain message unless the downloaded model files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} isn't downloaded yet. Run the ⬇️ Download cell above first.")


tissue = data.immunohistochemistry()
print("tissue", tissue.shape)

## 1. Scoring a mask: intersection over union

Before trusting any model's outlines you need a number for "how well does
this mask match the real one?" The standard one is **IoU**, intersection over
union: the pixels both masks mark, divided by the pixels either one marks.

In [ ]:
true_mask = np.zeros((10, 10), dtype=bool)
true_mask[2:7, 2:7] = True        # a 5 × 5 "cell"
guess = np.zeros((10, 10), dtype=bool)
guess[3:8, 3:8] = True            # the same size, shifted by one pixel down and right

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
axes[0].imshow(true_mask, cmap="gray"); axes[0].set_title("true")
axes[1].imshow(guess, cmap="gray"); axes[1].set_title("guess")
for ax in axes:
    ax.axis("off")
plt.show()

🔮 **Predict.** Count on the pictures: how many pixels do both masks share,
and how many does at least one of them mark? So what is the IoU?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 1a.** Write `iou(a, b)` for two True/False masks of the same
shape. `a & b` is True where both are; `a | b` where either is.

In [ ]:
def iou(a, b):
    return ...

In [ ]:
check(is_done(iou(true_mask, guess)), "iou still returns ... .")
check(np.isclose(iou(true_mask, guess), 16 / 34), "Both: a 4 × 4 overlap = 16. Either: 25 + 25 − 16 = 34.")
check(np.isclose(iou(true_mask, true_mask), 1.0), "A mask compared with itself should score 1.")
far = np.zeros((10, 10), dtype=bool)
far[8:10, 8:10] = True
check(np.isclose(iou(true_mask, far), 0.0), "Masks that don't touch should score 0.")

## 2. A model that finds nuclei

**Cellpose** is a network trained on thousands of microscope images to
outline cells and nuclei. It returns a **label image**: 0 for background, and
1, 2, 3 … for each nucleus it found, like Day 2's `ndimage.label` but drawn
by a model instead of a threshold.

⬇️ **Download the model (25 MB).** Run the next cell when you start section 2. It downloads the Cellpose `nuclei` model, 25 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: Cellpose nuclei model (25 MB)
from cellpose import models

CELLPOSE_FILES = [models.MODEL_DIR / "nucleitorch_0", models.MODEL_DIR / "size_nucleitorch_0.npy"]
if all(f.exists() for f in CELLPOSE_FILES):
    print("The Cellpose nuclei model is already downloaded. Nothing to do.")
else:
    models.model_path("nuclei")
    models.size_model_path("nuclei")
    print("Done: Cellpose nuclei model downloaded (25 MB).")

Big images are processed in tiles. Day 2's slicing does it.

In [ ]:
from cellpose import models

require_model("Cellpose nuclei model", models.MODEL_DIR / "nucleitorch_0", models.MODEL_DIR / "size_nucleitorch_0.npy")
segmenter = models.Cellpose(model_type="nuclei", gpu=False)


def get_tile(img, r, c, size):
    return img[r * size:(r + 1) * size, c * size:(c + 1) * size]


tile = get_tile(tissue, 1, 1, 256)
labels, *_ = segmenter.eval(tile, diameter=17, channels=[0, 0])
print("nuclei found in this tile:", labels.max())

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(tile); axes[0].set_title("tile (1, 1)")
axes[1].imshow(tile); axes[1].imshow(np.ma.masked_equal(labels, 0), cmap="tab20", alpha=0.6)
axes[1].set_title("Cellpose's nuclei")
for ax in axes:
    ax.axis("off")
plt.show()

That missed a lot: look at the dark nuclei inside the brown glands. The
`nuclei` model expects one channel in which nuclei stand out, and in the
colour photo the brown stain gets in the way.

`rgb2hed` separates the photo into its stains: **H**aematoxylin (the blue
nuclear stain), **E**osin, and **D**AB (the brown). Its first channel shows
only the nuclei.

In [ ]:
from skimage.color import rgb2hed


def nuclei_channel(img):
    """The haematoxylin (nuclear stain) channel, scaled to 0–255."""
    h = rgb2hed(img)[:, :, 0]
    return ((h - h.min()) / (h.max() - h.min()) * 255).astype(np.uint8)


hema = nuclei_channel(tissue)
plt.figure(figsize=(4, 4)); plt.imshow(hema, cmap="gray"); plt.title("haematoxylin only"); plt.axis("off"); plt.show()

🔮 **Predict.** Same model, same tile, but given the haematoxylin channel
instead of the colour photo. More nuclei, fewer, or the same? And Day 2's
threshold count for the whole image was around 90: will Cellpose's total on
the haematoxylin channel be higher or lower?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
labels_h, *_ = segmenter.eval(get_tile(hema, 1, 1, 256), diameter=17, channels=[0, 0])
print("colour photo:", labels.max(), "nuclei;  haematoxylin channel:", labels_h.max(), "nuclei")

all_labels = {}
for r in range(2):
    for c in range(2):
        all_labels[(r, c)], *_ = segmenter.eval(get_tile(hema, r, c, 256), diameter=17, channels=[0, 0])
        print(f"tile ({r}, {c}): {all_labels[(r, c)].max()} nuclei")
print("total:", sum(lab.max() for lab in all_labels.values()))

## 3. Trouble at the edges

Look at where the tiles meet. A nucleus sitting on the line between two
tiles is cut in two: each tile may count half of it as a whole nucleus.

✏️ **Exercise 3a.** Write `touching_edge(labels)`: how many different nuclei
in a label image touch its border (the first or last row, or the first or last
column)? Hint: collect the labels on the four edges with `np.concatenate`,
take `np.unique`, and don't count 0, which is background.

In [ ]:
def touching_edge(labels):
    return ...

In [ ]:
demo = np.zeros((6, 6), dtype=int)
demo[0, 2] = 1           # touches the top edge
demo[2:4, 2:4] = 2       # in the middle
demo[5, 0] = 3           # touches the bottom and left edges
check(is_done(touching_edge(demo)), "touching_edge still returns ... .")
check(touching_edge(demo) == 2, "In the demo, nuclei 1 and 3 touch an edge and nucleus 2 does not: 2.")
check(touching_edge(np.zeros((4, 4), dtype=int)) == 0, "With no nuclei the answer is 0: leave out the background label.")

In [ ]:
edge_total = sum(touching_edge(lab) for lab in all_labels.values())
total = sum(lab.max() for lab in all_labels.values())
print(f"{edge_total} of {total} nuclei touch a tile edge ({edge_total / total:.0%}).")

🤔 **Discuss.** Some of those edge nuclei are counted twice and some are only
half a nucleus. Project II's brief warns that detection is poor at tile edges.
Propose a fix with your partner. (Think: what if the tiles overlapped?)

## Extension (if you finish early)

- Cut overlapping tiles and count again. How much does the total change?
- Use `iou` to compare Cellpose's foreground (`labels > 0`) with Day 2's
  threshold mask on one tile. How much do they agree?
- Project II gives a centroid for every cell. Make up five centroids on a
  tile, and write a function that counts how many of them fall inside a
  Cellpose nucleus.

In [ ]:
# Your extension work here.